In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_ingestion import DataIngestion
from src.data_validation import DataValidation
from src.data_transformation import DataTransformation
from src.model_trainer import ModelTrainer
from src.utils import read_csv_safely

In [ ]:
ingestion = DataIngestion()
ingestion.config.raw_data_path = '../data/train.csv'
ingestion.config.train_data_path = '../artifacts/train.csv'
ingestion.config.test_data_path = '../artifacts/test.csv'

train_path, test_path = ingestion.initiate_data_ingestion()
print(train_path, test_path)

In [ ]:
train_df = read_csv_safely(train_path)
test_df = read_csv_safely(test_path)
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

In [ ]:
validator = DataValidation()
print('Train data valid:', validator.run_all_checks(train_df))
print('Test data valid:', validator.run_all_checks(test_df))

In [ ]:
transformer = DataTransformation(artifacts_dir='../artifacts')
X_train, y_train = transformer.fit_transform(train_df)
X_test = transformer.transform(test_df.drop(columns=['is_claim']))
y_test = test_df['is_claim']
print(X_train.shape, X_test.shape)

In [ ]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos
print(f'Positive samples: {pos}, Negative samples: {neg}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

In [ ]:
trainer = ModelTrainer(artifacts_dir='../artifacts')
best_name, best_model, results = trainer.train_and_evaluate(
    X_train, y_train, X_test, y_test
)

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)
results_df